In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]

def build_graph(model_name):
    tool = TavilySearchResults(max_results=2)
    tools = [tool]
    llm = ChatOpenAI(model_name=model_name)
    llm_with_tools = llm.bind_tools(tools)

    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    graph_builder = StateGraph(State)
    graph_builder.add_node("chatbot", chatbot)
    
    tool_node = ToolNode(tools)
    graph_builder.add_node("tools", tool_node)

    graph_builder.add_conditional_edges("chatbot", tools_condition)
    graph_builder.add_edge("tools", "chatbot")
    graph_builder.set_entry_point("chatbot")

    memory = MemorySaver()
    return graph_builder.compile(checkpointer=memory)

def stream_graph_updates(graph, user_input: str):
    config = {"configurable": {"thread_id": "1"}}
    events = graph.stream(
        {"messages": [("user", user_input)]}, 
        config, 
        stream_mode="values"
    )
    for event in events:
        print(event["messages"][-1].content, flush=True)

load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

MODEL_NAME = "gpt-4o-mini"

app_graph = build_graph(MODEL_NAME)

print("--- 会話開始、 終了するならEnterを入力ください。 ---")
while True:
    user_input = input("質問: ")
    if user_input.strip() == "":
        print("ありがとうございました!")
        break
    stream_graph_updates(app_graph, user_input)

--- 챗봇이 시작되었습니다. 종료하려면 엔터를 누르세요 ---
こんにちは！
こんにちは！どういったことをお手伝いできますか？
1たす2は？
1たす2は3です。何か他の質問がありますか？
台湾観光について検索結果を教えて

[{"url": "https://www.kkday.com/ja/blog/48762/taiwan-sightseeing?srsltid=AfmBOopV3karFJThFH5_W0z1f8kZYupGifDx7fuUKlUTgdW51zVLSDxS", "content": "台湾の観光前に知っておきたい基本情報\n  + 台湾の基本情報\n 台湾観光に必須！台北無限周遊パスを予約しよう\n 台湾・台北のおすすめ観光スポット16選\n  + 1. 台北101｜台北の街並みを一望できる台湾の観光名所\n  + 2. 龍山寺｜恋愛成就のパワースポットとして有名\n  + 3. 士林夜市｜台北で最大規模を誇る夜市\n  + 4. 国立故宮博物院｜世界屈指の大型博物館\n  + 5. 北投温泉｜台湾・台北市内から約40分でアクセスできる温泉街\n  + 6. 忠烈祠｜衛兵交代式が見どころ\n  + 7. 行天宮｜商業の神様が祀られている\n  + 8. 華山1914文化創意産業園区｜今と昔が融合した新感覚のアートスポット\n  + 9. 猫空｜台湾茶の名産地！ロープウェイからの絶景も堪能できる\n  + 10. 迪化街｜台湾のレトロな観光地！台北で人気のある問屋街\n  + 11. 中正紀念堂｜蒋介石を哀悼するために作られた\n  + 12. 饒河街観光夜市｜台湾グルメを堪能できるおすすめの夜市\n  + 13. 四四南村｜レトロな街並みが写真映えバッチリ\n  + 14. 剝皮寮歴史街区｜龍山寺近くのレトロな街並み\n  + 15. 象山｜台湾屈指の絶景スポット！美しい夜景も楽しめる\n  + 16. 台湾博物館鉄道部パーク｜台湾の鉄道について学べる観光スポット\n 台湾近郊のおすすめ観光スポット9選（新北・基隆）\n  + 17. 九份｜台湾で有名な観光名所！千と千尋の世界観を味わえる\n  + 18. 淡水｜台湾のヴェネツィア！美しい夕日が見られる\n  + 19. 野柳地質公園｜ユニークな奇岩が特徴的\n  + 20. 深澳レー